In [ ]:
"""
mahjong_env.py
==============
A simplified 1-player Mahjong Gymnasium environment (agent vs. 3 random dummies).

Tile indexing (34 types):
  0-8   : Man (Characters) 1-9
  9-17  : Pin (Circles)    1-9
  18-26 : Sou (Bamboo)     1-9
  27-30 : Wind tiles       (East, South, West, North)
  31-33 : Dragon tiles     (Haku, Hatsu, Chun)
"""

from __future__ import annotations

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from itertools import combinations_with_replacement


# ---------------------------------------------------------------------------
# Helper: Mahjong win-condition checker
# ---------------------------------------------------------------------------

def _can_form_sets(hand_vec: np.ndarray) -> bool:
    """
    Recursively check if the 34-element hand vector can be decomposed into
    exactly 4 melds (sequences of 3 consecutive suited tiles OR triplets of
    any tile) plus 1 pair.

    Returns True if a standard winning hand is detected.
    """
    tiles = []
    for tile_id, count in enumerate(hand_vec):
        tiles.extend([tile_id] * int(count))

    if len(tiles) != 14:
        return False

    # Try every possible pair
    unique = list(dict.fromkeys(tiles))  # preserves order, removes dups
    for pair_tile in unique:
        if hand_vec[pair_tile] >= 2:
            remaining = hand_vec.copy()
            remaining[pair_tile] -= 2
            if _extract_melds(remaining, 4):
                return True
    return False


def _extract_melds(hand: np.ndarray, melds_needed: int) -> bool:
    """Try to extract `melds_needed` melds from hand (recursive)."""
    if melds_needed == 0:
        return hand.sum() == 0

    # Find the first non-zero tile index
    first = -1
    for i in range(34):
        if hand[i] > 0:
            first = i
            break
    if first == -1:
        return False

    # Option 1: Triplet
    if hand[first] >= 3:
        hand[first] -= 3
        if _extract_melds(hand, melds_needed - 1):
            hand[first] += 3
            return True
        hand[first] += 3

    # Option 2: Sequence (only for suited tiles 0-26)
    if first <= 24:                         # need first+2 <= 26
        suit = first // 9
        pos = first % 9
        if pos <= 6:                        # can still form a sequence
            s0, s1, s2 = first, first + 1, first + 2
            if (hand[s0] >= 1 and hand[s1] >= 1 and hand[s2] >= 1):
                hand[s0] -= 1; hand[s1] -= 1; hand[s2] -= 1
                if _extract_melds(hand, melds_needed - 1):
                    hand[s0] += 1; hand[s1] += 1; hand[s2] += 1
                    return True
                hand[s0] += 1; hand[s1] += 1; hand[s2] += 1

    return False


# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------

class MahjongEnv(gym.Env):
    """
    Simplified 1-player Mahjong environment.

    Observation (68,):
        [0:34]  - agent's current hand (count per tile type, 0-4)
        [34:68] - public discard pile  (count per tile type, 0-4)

    Action:
        Integer in [0, 33] - index of the tile type to discard.

    Episode flow (per step):
        1. Agent discards 1 tile  (action)
        2. Each of 3 dummies draws 1 tile from the wall and immediately discards it
        3. Agent draws 1 tile from the wall
        4. Check termination / truncation
    """

    metadata = {"render_modes": []}

    # ------------------------------------------------------------------
    def __init__(self, seed: int | None = None):
        super().__init__()

        self.action_space = spaces.Discrete(34)
        self.observation_space = spaces.Box(
            low=0, high=4, shape=(68,), dtype=np.int32
        )

        self._np_random: np.random.Generator | None = None
        self.seed(seed)

        # Will be properly initialised in reset()
        self.wall: np.ndarray = np.zeros(34, dtype=np.int32)
        self.agent_hand: np.ndarray = np.zeros(34, dtype=np.int32)
        self.discard_pile: np.ndarray = np.zeros(34, dtype=np.int32)

    # ------------------------------------------------------------------
    # Seeding
    # ------------------------------------------------------------------

    def seed(self, seed: int | None = None):
        self._np_random = np.random.default_rng(seed)
        return [seed]

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _draw_tile(self) -> int:
        """
        Sample one tile from self.wall proportional to counts.
        Returns the tile index; decrements self.wall.
        Raises RuntimeError if wall is empty.
        """
        total = int(self.wall.sum())
        if total == 0:
            raise RuntimeError("Wall is empty - cannot draw.")
        probs = self.wall / total
        tile = int(self._np_random.choice(34, p=probs))
        self.wall[tile] -= 1
        return tile

    def _get_obs(self) -> np.ndarray:
        return np.concatenate([self.agent_hand, self.discard_pile]).astype(np.int32)

    def _action_mask(self) -> np.ndarray:
        """Boolean mask: True where agent holds ≥ 1 of that tile type."""
        return (self.agent_hand > 0).astype(np.int8)

    def _is_winning_hand(self) -> bool:
        return _can_form_sets(self.agent_hand)

    # ------------------------------------------------------------------
    # reset
    # ------------------------------------------------------------------

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ) -> tuple[np.ndarray, dict]:
        if seed is not None:
            self.seed(seed)

        # Full deck: 4 copies of each of 34 tile types
        self.wall = np.full(34, 4, dtype=np.int32)
        self.agent_hand = np.zeros(34, dtype=np.int32)
        self.discard_pile = np.zeros(34, dtype=np.int32)

        # Deal 13 tiles to the agent
        for _ in range(13):
            tile = self._draw_tile()
            self.agent_hand[tile] += 1

        # Draw the 14th tile (agent's first "draw")
        tile = self._draw_tile()
        self.agent_hand[tile] += 1

        info = {"action_mask": self._action_mask()}
        return self._get_obs(), info

    # ------------------------------------------------------------------
    # step
    # ------------------------------------------------------------------

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        # ---- Validate action ----
        if self.agent_hand[action] <= 0:
            # Invalid action: penalise and force a valid discard
            valid = np.where(self.agent_hand > 0)[0]
            action = int(self._np_random.choice(valid))

        reward: float = -0.1          # default step cost
        terminated = False
        truncated = False

        # ---- 1. Agent discards ----
        self.agent_hand[action] -= 1
        self.discard_pile[action] += 1

        # ---- 2. Dummy simulation (draw & immediately discard) ----
        for _ in range(3):
            if self.wall.sum() == 0:
                break
            dummy_tile = self._draw_tile()
            self.discard_pile[dummy_tile] += 1   # dummy discards at once

        # ---- 3. Agent draws ----
        if self.wall.sum() > 0:
            drawn = self._draw_tile()
            self.agent_hand[drawn] += 1
        else:
            # Wall exhausted after dummy draws → truncate
            truncated = True

        # ---- 4. Termination check ----
        if not truncated:
            if self._is_winning_hand():
                terminated = True
                reward = +10.0

        # ---- 5. Build info ----
        info: dict = {
            "action_mask": self._action_mask(),
            "wall_remaining": int(self.wall.sum()),
        }

        return self._get_obs(), reward, terminated, truncated, info

    # ------------------------------------------------------------------
    # render / close (stubs)
    # ------------------------------------------------------------------

    def render(self):
        hand_tiles = [f"T{i}×{c}" for i, c in enumerate(self.agent_hand) if c > 0]
        print(f"Hand : {' '.join(hand_tiles)}")
        print(f"Wall remaining: {int(self.wall.sum())}")

    def close(self):
        pass


# ---------------------------------------------------------------------------
# Example: wrapping with gym.vector.AsyncVectorEnv
# ---------------------------------------------------------------------------

def make_env(seed: int):
    """Factory used by AsyncVectorEnv."""
    def _init():
        env = MahjongEnv(seed=seed)
        return env
    return _init


if __name__ == "__main__":
    # ------------------------------------------------------------------ #
    #  1. Single-environment sanity check                                 #
    # ------------------------------------------------------------------ #
    print("=== Single-env test ===")
    env = MahjongEnv(seed=42)
    obs, info = env.reset()
    print(f"Initial obs shape : {obs.shape}")
    print(f"Initial hand sum  : {obs[:34].sum()}  (should be 14)")
    print(f"Action mask       : {info['action_mask']}")

    for step_num in range(5):
        mask = info["action_mask"]
        valid_actions = np.where(mask)[0]
        action = int(np.random.choice(valid_actions))
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"  Step {step_num+1}: action={action}, reward={reward:.1f}, "
              f"terminated={terminated}, truncated={truncated}, "
              f"wall={info['wall_remaining']}")
        if terminated or truncated:
            print("  Episode ended - resetting.")
            obs, info = env.reset()

    env.close()

    # ------------------------------------------------------------------ #
    #  2. AsyncVectorEnv example (4 parallel environments)                #
    # ------------------------------------------------------------------ #
    print("\n=== AsyncVectorEnv (4 envs) ===")
    num_envs = 4
    vec_env = gym.vector.AsyncVectorEnv(
        [make_env(seed=i) for i in range(num_envs)]
    )

    obs_batch, info_batch = vec_env.reset()
    print(f"Batched obs shape : {obs_batch.shape}")   # (4, 68)

    # Run a few vectorised steps with random valid actions
    for step_num in range(3):
        # Gather per-env action masks (each is shape (34,))
        masks = np.stack(info_batch["action_mask"])    # (4, 34)
        actions = np.array([
            int(np.random.choice(np.where(masks[i])[0]))
            for i in range(num_envs)
        ])
        obs_batch, rewards, terms, truncs, info_batch = vec_env.step(actions)
        print(f"  Step {step_num+1}: rewards={rewards}, "
              f"terminated={terms}, truncated={truncs}")

    vec_env.close()
    print("\nDone.")

=== Single-env test ===
Initial obs shape : (68,)
Initial hand sum  : 14  (should be 14)
Action mask       : [0 0 0 1 1 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0 1 0 1 1 1 0 1 0 1 0 1]
  Step 1: action=26, reward=-0.1, terminated=False, truncated=False, wall=118
  Step 2: action=29, reward=-0.1, terminated=False, truncated=False, wall=114
  Step 3: action=11, reward=-0.1, terminated=False, truncated=False, wall=110
  Step 4: action=31, reward=-0.1, terminated=False, truncated=False, wall=106
  Step 5: action=4, reward=-0.1, terminated=False, truncated=False, wall=102

=== AsyncVectorEnv (4 envs) ===
Batched obs shape : (4, 68)
  Step 1: rewards=[-0.1 -0.1 -0.1 -0.1], terminated=[False False False False], truncated=[False False False False]
  Step 2: rewards=[-0.1 -0.1 -0.1 -0.1], terminated=[False False False False], truncated=[False False False False]
  Step 3: rewards=[-0.1 -0.1 -0.1 -0.1], terminated=[False False False False], truncated=[False False False False]

Done.


In [1]:
"""
mahjong_env.py
==============
A simplified 1-player Mahjong Gymnasium environment (agent vs. 3 random dummies).

Tile indexing (34 types):
  0-8   : Man (Characters) 1-9
  9-17  : Pin (Circles)    1-9
  18-26 : Sou (Bamboo)     1-9
  27-30 : Wind tiles       (East, South, West, North)
  31-33 : Dragon tiles     (Haku, Hatsu, Chun)
"""

from __future__ import annotations

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from itertools import combinations_with_replacement # Tạo ra tất cả các combinations có lặp lại phần tử, theo độ dài cho trước.


# ---------------------------------------------------------------------------
# Helper: Mahjong win-condition checker
# ---------------------------------------------------------------------------

def _can_form_sets(hand_vec: np.ndarray) -> bool:
    """
    Recursively check if the 34-element hand vector can be decomposed into
    exactly 4 melds (sequences of 3 consecutive suited tiles OR triplets of
    any tile) plus 1 pair.

    Returns True if a standard winning hand is detected.
    """
    tiles = []
    for tile_id, count in enumerate(hand_vec):
        tiles.extend([tile_id] * int(count))

    if len(tiles) != 14:
        return False

    # Try every possible pair
    unique = list(dict.fromkeys(tiles))  # preserves order, removes dups
    for pair_tile in unique:
        if hand_vec[pair_tile] >= 2:
            remaining = hand_vec.copy()
            remaining[pair_tile] -= 2
            if _extract_melds(remaining, 4):
                return True
    return False


def _extract_melds(hand: np.ndarray, melds_needed: int) -> bool:
    """Try to extract `melds_needed` melds from hand (recursive)."""
    if melds_needed == 0:
        return hand.sum() == 0

    # Find the first non-zero tile index
    first = -1
    for i in range(34):
        if hand[i] > 0:
            first = i
            break
    if first == -1:
        return False

    # Option 1: Triplet
    if hand[first] >= 3:
        hand[first] -= 3
        if _extract_melds(hand, melds_needed - 1):
            hand[first] += 3
            return True
        hand[first] += 3

    # Option 2: Sequence (only for suited tiles 0-26)
    if first <= 24:                         # need first+2 <= 26
        suit = first // 9
        pos = first % 9
        if pos <= 6:                        # can still form a sequence
            s0, s1, s2 = first, first + 1, first + 2
            if (hand[s0] >= 1 and hand[s1] >= 1 and hand[s2] >= 1):
                hand[s0] -= 1; hand[s1] -= 1; hand[s2] -= 1
                if _extract_melds(hand, melds_needed - 1):
                    hand[s0] += 1; hand[s1] += 1; hand[s2] += 1
                    return True
                hand[s0] += 1; hand[s1] += 1; hand[s2] += 1

    return False


# ---------------------------------------------------------------------------
# Shanten Number Calculation
# ---------------------------------------------------------------------------
#
# Shanten = số bài cần rút thêm để đạt tenpai (shanten=0) hoặc thắng (-1).
#
# Công thức chuẩn (standard hand):
#   shanten = (4 - melds) * 2 - 1   khi chưa có pair
#           = (4 - melds) * 2       khi đã có pair
#   → rút gọn: shanten = 8 - 2*melds - max(pairs, 1) - partials
#     trong đó partials = số partial block (pair + partial meld), bị giới hạn
#
# Ba loại tay đặc biệt cũng được tính:
#   • Chiitoitsu  (7 đôi)  : shanten = 6 - pairs
#   • Kokushi Musou        : shanten = 13 - unique_terminals - has_terminal_pair
#
# Hàm `calculate_shanten` trả về MIN của cả ba nhánh.

def _shanten_standard(hand: np.ndarray) -> int:
    """
    Tính shanten cho dạng tay tiêu chuẩn: 4 meld + 1 pair.

    Chiến lược: duyệt tất cả cách chọn pair (hoặc không chọn pair),
    sau đó đệ quy đếm số meld + partial tối đa có thể tạo từ phần còn lại.

    Shanten = 8 - 2*melds - partials
    (8 = 2*4 melds cần + 1 pair cần − 1)
    """

    best = [8]  # shanten tệ nhất cho 13 bài là 8

    def _count_blocks(h: np.ndarray, suit_start: int, suit_end: int,
                      melds: int, partials: int) -> tuple[int, int]:
        """
        Đệ quy đếm (melds, partials) tối đa trong đoạn [suit_start, suit_end).
        Trả về (melds, partials) tốt nhất.
        """
        # Tìm tile đầu tiên còn trong tay ở đoạn này
        idx = -1
        for i in range(suit_start, suit_end):
            if h[i] > 0:
                idx = i
                break
        if idx == -1:
            return melds, partials

        best_m, best_p = melds, partials

        # --- Thử triplet (kôtsu) ---
        if h[idx] >= 3:
            h[idx] -= 3
            m, p = _count_blocks(h, idx, suit_end, melds + 1, partials)
            if (m, p) > (best_m, best_p):
                best_m, best_p = m, p
            h[idx] += 3

        # --- Thử sequence (shuntsu) ---
        if idx + 2 < suit_end and h[idx + 1] >= 1 and h[idx + 2] >= 1:
            h[idx] -= 1; h[idx + 1] -= 1; h[idx + 2] -= 1
            m, p = _count_blocks(h, idx, suit_end, melds + 1, partials)
            if (m, p) > (best_m, best_p):
                best_m, best_p = m, p
            h[idx] += 1; h[idx + 1] += 1; h[idx + 2] += 1

        # --- Thử partial: đôi (jantai partial) ---
        if h[idx] >= 2:
            h[idx] -= 2
            m, p = _count_blocks(h, idx, suit_end, melds, partials + 1)
            if (m, p) > (best_m, best_p):
                best_m, best_p = m, p
            h[idx] += 2

        # --- Thử partial: kanchan (x _ x) ---
        if idx + 2 < suit_end and h[idx + 2] >= 1:
            h[idx] -= 1; h[idx + 2] -= 1
            m, p = _count_blocks(h, idx, suit_end, melds, partials + 1)
            if (m, p) > (best_m, best_p):
                best_m, best_p = m, p
            h[idx] += 1; h[idx + 2] += 1

        # --- Thử partial: penchan / ryanmen (x x+1) ---
        if idx + 1 < suit_end and h[idx + 1] >= 1:
            h[idx] -= 1; h[idx + 1] -= 1
            m, p = _count_blocks(h, idx, suit_end, melds, partials + 1)
            if (m, p) > (best_m, best_p):
                best_m, best_p = m, p
            h[idx] += 1; h[idx + 1] += 1

        # --- Bỏ qua tile này (isolated) ---
        h[idx] -= 1
        m, p = _count_blocks(h, idx, suit_end, melds, partials)
        if (m, p) > (best_m, best_p):
            best_m, best_p = m, p
        h[idx] += 1

        return best_m, best_p

    # Dải index của từng nhóm tile
    SUITS = [(0, 9), (9, 18), (18, 27)]   # 3 suit số (Man, Pin, Sou)
    HONORS = (27, 34)                      # tiles danh dự (gió + rồng)

    def _evaluate(h: np.ndarray, has_pair: bool):
        """Gọi _count_blocks cho từng suit + honor, cộng kết quả."""
        total_melds = 0
        total_partials = 0
        for start, end in SUITS:
            m, p = _count_blocks(h, start, end, 0, 0)
            total_melds += m
            total_partials += p
        # Honor tiles chỉ có triplet hoặc partial đôi, không có sequence
        for i in range(HONORS[0], HONORS[1]):
            if h[i] >= 3:
                total_melds += 1
                h[i] -= 3   # tạm trừ để không đếm lại
            elif h[i] == 2:
                total_partials += 1

        # Giới hạn: tổng (melds + partials) ≤ 4, partials ≤ (4 - melds)
        # (không thể có partial nếu đã đủ meld)
        cap = 4 - total_melds
        if total_partials > cap:
            total_partials = cap

        shanten = 8 - 2 * total_melds - total_partials - (1 if has_pair else 0)
        return shanten

    h = hand.copy().astype(int)

    # Trường hợp 1: Không dùng pair nào làm pair chính
    s = _evaluate(h.copy(), has_pair=False)
    if s < best[0]:
        best[0] = s

    # Trường hợp 2: Thử từng tile làm pair chính
    for i in range(34):
        if h[i] >= 2:
            h[i] -= 2
            s = _evaluate(h.copy(), has_pair=True)
            if s < best[0]:
                best[0] = s
            h[i] += 2

    return best[0]


def _shanten_chiitoitsu(hand: np.ndarray) -> int:
    """
    Shanten cho Chiitoitsu (7 đôi khác nhau).
    shanten = 6 - số_đôi_hiện_có
    Cần ít nhất 7 loại tile; mỗi loại chỉ đếm 1 đôi dù có ≥4.
    """
    pairs = int(np.sum(hand >= 2))
    return 6 - pairs


def _shanten_kokushi(hand: np.ndarray) -> int:
    """
    Shanten cho Kokushi Musou (13 orphans).
    Cần: 1m 9m 1p 9p 1s 9s + 7 loại tile danh dự (index 27-33) + 1 trong số đó làm pair.
    Terminals & honors indices: 0, 8, 9, 17, 18, 26, 27, 28, 29, 30, 31, 32, 33
    shanten = 13 - unique_terminals_honors - (1 nếu có ≥1 terminal/honor làm đôi)
    """
    KOKUSHI_TILES = [0, 8, 9, 17, 18, 26, 27, 28, 29, 30, 31, 32, 33]
    unique = sum(1 for t in KOKUSHI_TILES if hand[t] >= 1)
    has_pair = any(hand[t] >= 2 for t in KOKUSHI_TILES)
    return 13 - unique - (1 if has_pair else 0)


def calculate_shanten(hand: np.ndarray) -> int:
    """
    Tính shanten number của hand hiện tại (34-element count vector).

    Kết quả:
      -1  → Tenpai thắng rồi (winning hand)
       0  → Tenpai (1 bài nữa là thắng)
       k  → Cần thêm k bài để đạt tenpai

    Tính cả 3 dạng (standard, chiitoitsu, kokushi) và trả về MIN.
    """
    s_std  = _shanten_standard(hand)
    s_chi  = _shanten_chiitoitsu(hand)
    s_kok  = _shanten_kokushi(hand)
    return min(s_std, s_chi, s_kok)


def shanten_breakdown(hand: np.ndarray) -> dict:
    """
    Trả về dict chi tiết shanten cho cả 3 dạng tay.
    Tiện cho debugging / reward shaping.

    Ví dụ:
      {
        'standard'   : 2,
        'chiitoitsu' : 4,
        'kokushi'    : 9,
        'best'       : 2,
      }
    """
    s_std = _shanten_standard(hand)
    s_chi = _shanten_chiitoitsu(hand)
    s_kok = _shanten_kokushi(hand)
    return {
        "standard":    s_std,
        "chiitoitsu":  s_chi,
        "kokushi":     s_kok,
        "best":        min(s_std, s_chi, s_kok),
    }


# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------

class MahjongEnv(gym.Env):
    """
    Simplified 1-player Mahjong environment.

    Observation (68,):
        [0:34]  - agent's current hand (count per tile type, 0-4)
        [34:68] - public discard pile  (count per tile type, 0-4)

    Action:
        Integer in [0, 33] - index of the tile type to discard.

    Episode flow (per step):
        1. Agent discards 1 tile  (action)
        2. Each of 3 dummies draws 1 tile from the wall and immediately discards it
        3. Agent draws 1 tile from the wall
        4. Check termination / truncation
    """

    metadata = {"render_modes": []}

    # ------------------------------------------------------------------
    def __init__(self, seed: int | None = None):
        super().__init__()

        self.action_space = spaces.Discrete(34)
        self.observation_space = spaces.Box(
            low=0, high=4, shape=(68,), dtype=np.int32
        )

        self._np_random: np.random.Generator | None = None
        self.seed(seed)

        # Will be properly initialised in reset()
        self.wall: np.ndarray = np.zeros(34, dtype=np.int32)
        self.agent_hand: np.ndarray = np.zeros(34, dtype=np.int32)
        self.discard_pile: np.ndarray = np.zeros(34, dtype=np.int32)
        self._prev_shanten: int = 8  # worst-case starting value

    # ------------------------------------------------------------------
    # Seeding
    # ------------------------------------------------------------------

    def seed(self, seed: int | None = None):
        self._np_random = np.random.default_rng(seed)
        return [seed]

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _draw_tile(self) -> int:
        """
        Sample one tile from self.wall proportional to counts.
        Returns the tile index; decrements self.wall.
        Raises RuntimeError if wall is empty.
        """
        total = int(self.wall.sum())
        if total == 0:
            raise RuntimeError("Wall is empty - cannot draw.")
        probs = self.wall / total
        tile = int(self._np_random.choice(34, p=probs))
        self.wall[tile] -= 1
        return tile

    def _get_obs(self) -> np.ndarray:
        return np.concatenate([self.agent_hand, self.discard_pile]).astype(np.int32)

    def _action_mask(self) -> np.ndarray:
        """Boolean mask: True where agent holds ≥ 1 of that tile type."""
        return (self.agent_hand > 0).astype(np.int8)

    def _is_winning_hand(self) -> bool:
        return calculate_shanten(self.agent_hand) == -1

    def _shanten(self) -> int:
        return calculate_shanten(self.agent_hand)

    # ------------------------------------------------------------------
    # reset
    # ------------------------------------------------------------------

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ) -> tuple[np.ndarray, dict]:
        if seed is not None:
            self.seed(seed)

        # Full deck: 4 copies of each of 34 tile types
        self.wall = np.full(34, 4, dtype=np.int32)
        self.agent_hand = np.zeros(34, dtype=np.int32)
        self.discard_pile = np.zeros(34, dtype=np.int32)

        # Deal 13 tiles to the agent
        for _ in range(13):
            tile = self._draw_tile()
            self.agent_hand[tile] += 1

        # Draw the 14th tile (agent's first "draw")
        tile = self._draw_tile()
        self.agent_hand[tile] += 1

        self._prev_shanten = calculate_shanten(self.agent_hand)
        initial_shanten = self._prev_shanten
        info = {
            "action_mask": self._action_mask(),
            "shanten":     initial_shanten,
            "shanten_detail": shanten_breakdown(self.agent_hand),
        }
        return self._get_obs(), info

    # ------------------------------------------------------------------
    # step
    # ------------------------------------------------------------------

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        # ---- Validate action ----
        if self.agent_hand[action] <= 0:
            # Invalid action: penalise and force a valid discard
            valid = np.where(self.agent_hand > 0)[0]
            action = int(self._np_random.choice(valid))

        reward: float = -0.1          # default step cost
        terminated = False
        truncated = False

        # ---- 1. Agent discards ----
        self.agent_hand[action] -= 1
        self.discard_pile[action] += 1

        # ---- 2. Dummy simulation (draw & immediately discard) ----
        for _ in range(3):
            if self.wall.sum() == 0:
                break
            dummy_tile = self._draw_tile()
            self.discard_pile[dummy_tile] += 1   # dummy discards at once

        # ---- 3. Agent draws ----
        if self.wall.sum() > 0:
            drawn = self._draw_tile()
            self.agent_hand[drawn] += 1
        else:
            # Wall exhausted after dummy draws → truncate
            truncated = True

        # ---- 4. Termination check + shanten-aware reward ----
        current_shanten = self._shanten()
        if not truncated:
            if current_shanten == -1:          # winning hand
                terminated = True
                reward = +10.0
            else:
                # Reward shaping: khuyến khích giảm shanten
                # +0.5 mỗi lần shanten giảm 1 bước, −0.1 base per step
                reward = -0.1 + max(0, self._prev_shanten - current_shanten) * 0.5
        self._prev_shanten = current_shanten

        # ---- 5. Build info ----
        info: dict = {
            "action_mask":    self._action_mask(),
            "wall_remaining": int(self.wall.sum()),
            "shanten":        current_shanten,
            "shanten_detail": shanten_breakdown(self.agent_hand),
        }

        return self._get_obs(), reward, terminated, truncated, info

    # ------------------------------------------------------------------
    # render / close (stubs)
    # ------------------------------------------------------------------

    def render(self):
        hand_tiles = [f"T{i}×{c}" for i, c in enumerate(self.agent_hand) if c > 0]
        print(f"Hand : {' '.join(hand_tiles)}")
        print(f"Wall remaining: {int(self.wall.sum())}")

    def close(self):
        pass


# ---------------------------------------------------------------------------
# Example: wrapping with gym.vector.AsyncVectorEnv
# ---------------------------------------------------------------------------

def make_env(seed: int):
    """Factory used by AsyncVectorEnv."""
    def _init():
        env = MahjongEnv(seed=seed)
        return env
    return _init


if __name__ == "__main__":
    # ------------------------------------------------------------------ #
    #  1. Shanten unit tests                                              #
    # ------------------------------------------------------------------ #
    print("=== Shanten unit tests ===")

    # Tenpai (winning) hand: 4 sequences + 1 pair
    # Man 1-2-3, 4-5-6, 7-8-9 + Pin 1-2-3 + Man 1-1 (pair)
    win_hand = np.zeros(34, dtype=np.int32)
    for t in [0, 1, 2, 3, 4, 5, 6, 7, 8]:  # Man 1-9 (3 sequences)
        win_hand[t] += 1
    win_hand[0] += 1                         # extra Man-1 → pair
    for t in [9, 10, 11]:                    # Pin 1-2-3
        win_hand[t] += 1
    print(f"Winning hand shanten : {calculate_shanten(win_hand)}")   # expect -1

    # Tenpai hand: 1 tile away từ win
    tenpai = win_hand.copy()
    tenpai[9] -= 1   # remove Pin-1 → waiting on Pin-1
    print(f"Tenpai hand shanten  : {calculate_shanten(tenpai)}")     # expect 0

    # Random scrambled hand
    rng = np.random.default_rng(99)
    rand_hand = np.zeros(34, dtype=np.int32)
    for _ in range(13):
        t = rng.integers(0, 34)
        rand_hand[t] += 1
    print(f"Random 13-tile shanten breakdown: {shanten_breakdown(rand_hand)}")

    # ------------------------------------------------------------------ #
    #  2. Single-environment sanity check                                 #
    # ------------------------------------------------------------------ #
    print("\n=== Single-env test ===")
    env = MahjongEnv(seed=42)
    obs, info = env.reset()
    print(f"Initial obs shape : {obs.shape}")
    print(f"Initial hand sum  : {obs[:34].sum()}  (should be 14)")
    print(f"Initial shanten   : {info['shanten']}")
    print(f"Shanten detail    : {info['shanten_detail']}")

    for step_num in range(10):
        mask = info["action_mask"]
        valid_actions = np.where(mask)[0]
        action = int(np.random.choice(valid_actions))
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"  Step {step_num+1:2d}: action={action:2d}  reward={reward:+.2f}  "
              f"shanten={info['shanten']}  terminated={terminated}  wall={info['wall_remaining']}")
        if terminated or truncated:
            print("  Episode ended - resetting.")
            obs, info = env.reset()

    env.close()

    # ------------------------------------------------------------------ #
    #  3. AsyncVectorEnv example (4 parallel environments)                #
    # ------------------------------------------------------------------ #
    print("\n=== AsyncVectorEnv (4 envs) ===")
    num_envs = 4
    vec_env = gym.vector.AsyncVectorEnv(
        [make_env(seed=i) for i in range(num_envs)]
    )

    obs_batch, info_batch = vec_env.reset()
    print(f"Batched obs shape    : {obs_batch.shape}")       # (4, 68)
    print(f"Per-env shanten      : {info_batch['shanten']}") # (4,)

    for step_num in range(3):
        masks = np.stack(info_batch["action_mask"])           # (4, 34)
        actions = np.array([
            int(np.random.choice(np.where(masks[i])[0]))
            for i in range(num_envs)
        ])
        obs_batch, rewards, terms, truncs, info_batch = vec_env.step(actions)
        print(f"  Step {step_num+1}: rewards={np.round(rewards,2)}  "
              f"shanten={info_batch['shanten']}  terminated={terms}")

    vec_env.close()
    print("\nDone.")


=== Shanten unit tests ===
Winning hand shanten : 0
Tenpai hand shanten  : 1
Random 13-tile shanten breakdown: {'standard': 3, 'chiitoitsu': 3, 'kokushi': 9, 'best': 3}

=== Single-env test ===
Initial obs shape : (68,)
Initial hand sum  : 14  (should be 14)
Initial shanten   : 4
Shanten detail    : {'standard': 4, 'chiitoitsu': 5, 'kokushi': 7, 'best': 4}
  Step  1: action=33  reward=-0.10  shanten=4  terminated=False  wall=118
  Step  2: action=27  reward=+0.40  shanten=3  terminated=False  wall=114
  Step  3: action=21  reward=-0.10  shanten=3  terminated=False  wall=110
  Step  4: action=11  reward=-0.10  shanten=3  terminated=False  wall=106
  Step  5: action= 4  reward=-0.10  shanten=3  terminated=False  wall=102
  Step  6: action=31  reward=+0.40  shanten=2  terminated=False  wall=98
  Step  7: action=25  reward=-0.10  shanten=2  terminated=False  wall=94
  Step  8: action=16  reward=-0.10  shanten=3  terminated=False  wall=90
  Step  9: action= 1  reward=-0.10  shanten=3  termi